In [1]:
# Désinstallation de la librairie transformers pour éviter les conflits
!pip uninstall -y transformers accelerate bitsandbytes

# Installation de la librairie OpenAI (officielle) et des outils audio
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q pydub
!apt update && apt install -y ffmpeg

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu

In [ ]:
import whisper
import torch
from pydub import AudioSegment
import re
import os
import math

# ----------------- CONFIGURATION - MODÈLE LARGE & SEGMENTS -----------------
# **CHANGEMENT : Utilisation du modèle 'large' (Qualité maximale, risque de plantage élevé sur CPU/GPU faible)**
audio_file_path = "../public/audio/gabrielle/Majorite_de_minorite.mp3"
model_name = "medium"
language = "fr"
song_id = 1
# ---------------------------------------------------------------------------

# --- 1. Vérification et Préparation du Fichier Audio ---
temp_wav_path = "temp_audio.wav"
try:
    print(f"Chargement de l'audio: {audio_file_path}...")
    audio = AudioSegment.from_file(audio_file_path)
    audio.export(temp_wav_path, format="wav")
    print(f"Fichier converti temporairement en {temp_wav_path}")
except Exception as e:
    print(f"Erreur lors du chargement ou de la conversion de l'audio: {e}")
    exit()

# --- 2. Chargement du Modèle et Transcriptions ---

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Chargement du modèle Whisper '{model_name}' sur {device}...")

model = whisper.load_model(model_name, device=device)

print("Démarrage de la transcription (Par segments)...")

# Exécution SANS word_timestamps=True pour obtenir l'horodatage par segment
transcription_result = model.transcribe(
    temp_wav_path,
    language=language,
    word_timestamps=False, # Désactive l'horodatage par mot
    verbose=False
)

print("Transcription terminée.")

# --- 3. Formatage de la Sortie ---

def format_for_karaoke_by_segment(transcription_result, song_id):
    """
    Formate la sortie en utilisant la structure 'segments' native de la librairie OpenAI
    pour garantir une ligne par segment/phrase.
    """
    lyrics_lines = []

    # La sortie 'segments' contient l'horodatage de début (start) et le texte pour chaque segment
    for segment in transcription_result['segments']:

        start_time_float = segment['start']
        start_time_seconds = math.floor(start_time_float)

        # Le texte du segment, nettoyé et sans apostrophes non échappées
        text = segment['text'].strip()
        text = text.replace("'", "\\'")

        if text and re.search(r'\w', text):
             lyrics_lines.append(
                f"{{ time: {start_time_seconds}, text: '{text}' }}"
            )

    if not lyrics_lines:
        return "Erreur: Aucune ligne de parole n'a pu être extraite des segments."

    # Construction de la chaîne JS finale
    js_output = f"""const localLyricsData: Record<number, LyricLine[]> = {{
  {song_id}: [
    {{ time: 0, text: '...' }},
    {"\n    ".join(lyrics_lines)},
  ],
}};"""

    return js_output

# Générer la sortie formatée
karaoke_js_output = format_for_karaoke_by_segment(transcription_result, song_id)


# --- 4. Affichage du Résultat ---
print("\n" + "="*50)
print("             ✅ RÉSULTAT AU FORMAT JAVASCRIPT ✅")
print("="*50)
print(karaoke_js_output)
print("="*50 + "\n")

# Nettoyage
if os.path.exists(temp_wav_path):
    os.remove(temp_wav_path)
    print(f"Fichier temporaire {temp_wav_path} supprimé.")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Chargement de l'audio: ../public/audio/gabrielle/Majorite_de_minorite.mp3...
Erreur lors du chargement ou de la conversion de l'audio: [Errno 2] No such file or directory: '../public/audio/gabrielle/Majorite_de_minorite.mp3'
Chargement du modèle Whisper 'medium' sur cpu...


100%|█████████████████████████████████████| 1.42G/1.42G [00:52<00:00, 29.3MiB/s]


Démarrage de la transcription (Par segments)...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


RuntimeError: Failed to load audio: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --ena

: 